In [25]:
import pandas as pd
df=pd.read_csv('cleaned_data.csv')

print(df.isnull().sum())
print(df.duplicated().sum())
df.sample(10)




answers    0
label      0
dtype: int64
0


,answers,label
44167,"[""It is common for companies to offer employee...",1
42529,"['As an artificial intelligence, I do not enga...",1
14189,"['Adobe develops Flash , and it also created t...",0
1704,"[""Companies pay for promoted trends and promot...",0
27755,"[""\\nAvoiding left turns can save gas because ...",1
13391,"['Peel an orange . Now , lay it flat on a tabl...",0
20205,"[""Firstly, I highly doubt anyone on this site ...",0
23002,"['In the animal kingdom, it is common for cert...",1
24086,['Tilt-shift photography is a technique that i...,1
22932,"[""During the State of the Union address, the p...",1


In [26]:
import ast

def clean_text(x):
    if isinstance(x, list):
        return " ".join(map(str, x))

    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)

            if isinstance(parsed, list):
                return " ".join(map(str, parsed))
            else:
                return str(parsed)

        except:
            return x   # IMPORTANT: fallback safe

    return str(x)



In [27]:
df['answers'] = df['answers'].str.lower()
df['answers']=df['answers'].apply(clean_text)

df.head()

,answers,label
0,"basically there are many categories of "" best ...",0
1,salt is good for not dying in car crashes and ...,0
2,the way it works is that old tv stations got a...,0
3,you ca n't just go around assassinating the le...,0
4,wanting to kill the shit out of germans drives...,0


In [28]:
df_feat = df.copy()
df_feat.sample(5)

,answers,label
17366,text book values drop rather rapidly and fluct...,0
34664,hope solo is a former professional soccer play...,1
42986,there are a few reasons why stocks and options...,1
3323,there are six stories . in each of the stories...,0
1478,"a 53 ' trailer can hold over 3,500 square feet...",0


In [29]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [31]:
def load_word_list(path):
    with open(path, "r", encoding="utf-8") as f:
        return set(line.strip().lower() for line in f if line.strip())

chat_words = load_word_list("chat_words.txt")
function_words = load_word_list("function_words.txt")
discourse_markers = load_word_list("discourse_markers.txt")

In [ ]:
def extract_features(text):
        doc = nlp(text)

            tokens = [t.text.lower() for t in doc if not t.is_space]
                alpha_tokens = [t.text.lower() for t in doc if t.is_alpha]

                    total_tokens = len(tokens)
                        total_alpha = len(alpha_tokens)

                            if total_tokens == 0 or total_alpha == 0:
                                    return {
                                                "chat_word_count": 0,
                                                            "punct_count": 0,
                                                                        "spelling_error_ratio": 0,
                                                                                    "ttr": 0,
                                                                                                "function_word_ratio": 0,
                                                                                                            "discourse_ratio": 0,
                                                                                                                        "function_count": 0,
                                                                                                                                    "discourse_count": 0,
                                                                                                                                                "sentence_length": 0,
                                                                                                                                                            "avg_word_length": 0,
                                                                                                                                                                        # Interaction terms
                                                                                                                                                                                    "function_count_x_length": 0,
                                                                                                                                                                                                "discourse_count_x_length": 0,
                                                                                                                                                                                                            "function_ratio_x_length": 0,
                                                                                                                                                                                                                        "discourse_ratio_x_length": 0,
                                                                                                                                                                                                                                }

                                                                                                                                                                                                                                    # ---------------- CHAT WORDS ----------------
                                                                                                                                                                                                                                        chat_word_count = sum(1 for t in tokens if t in chat_words)

                                                                                                                                                                                                                                            # ---------------- PUNCTUATION ----------------
                                                                                                                                                                                                                                                punct_count = sum(1 for t in doc if t.is_punct)

                                                                                                                                                                                                                                                    # ---------------- SPELLING ERRORS ----------------
                                                                                                                                                                                                                                                        spelling_errors = sum(1 for t in doc if t.is_alpha and t.is_oov)
                                                                                                                                                                                                                                                            spelling_error_ratio = spelling_errors / total_alpha

                                                                                                                                                                                                                                                                # ---------------- TYPE TOKEN RATIO ----------------
                                                                                                                                                                                                                                                                    ttr = len(set(alpha_tokens)) / total_alpha

                                                                                                                                                                                                                                                                        # ---------------- FUNCTION WORDS ----------------
                                                                                                                                                                                                                                                                            function_count = sum(1 for t in tokens if t in function_words)
                                                                                                                                                                                                                                                                                function_word_ratio = function_count / total_tokens

                                                                                                                                                                                                                                                                                    # ---------------- DISCOURSE MARKERS ----------------
                                                                                                                                                                                                                                                                                        discourse_count = sum(1 for t in tokens if t in discourse_markers)
                                                                                                                                                                                                                                                                                            discourse_ratio = discourse_count / total_tokens

                                                                                                                                                                                                                                                                                                # ---------------- META ----------------
                                                                                                                                                                                                                                                                                                    sentence_length = total_tokens
                                                                                                                                                                                                                                                                                                        avg_word_length = sum(len(t) for t in alpha_tokens) / total_alpha

                                                                                                                                                                                                                                                                                                            # ---------------- INTERACTION TERMS ----------------
                                                                                                                                                                                                                                                                                                                # These are the key addition — they let logistic regression learn:
                                                                                                                                                                                                                                                                                                                    # "when sentence is SHORT, trust the counts"
                                                                                                                                                                                                                                                                                                                        # "when sentence is LONG, trust the ratios"
                                                                                                                                                                                                                                                                                                                            function_count_x_length = function_count * sentence_length
                                                                                                                                                                                                                                                                                                                                discourse_count_x_length = discourse_count * sentence_length
                                                                                                                                                                                                                                                                                                                                    function_ratio_x_length = function_word_ratio * sentence_length
                                                                                                                                                                                                                                                                                                                                        discourse_ratio_x_length = discourse_ratio * sentence_length

                                                                                                                                                                                                                                                                                                                                            return {
                                                                                                                                                                                                                                                                                                                                                    # Raw counts — reliable for short sentences
                                                                                                                                                                                                                                                                                                                                                            "chat_word_count": chat_word_count,
                                                                                                                                                                                                                                                                                                                                                                    "punct_count": punct_count,
                                                                                                                                                                                                                                                                                                                                                                            "function_count": function_count,
                                                                                                                                                                                                                                                                                                                                                                                    "discourse_count": discourse_count,
                                                                                                                                                                                                                                                                                                                                                                                            # Ratios — reliable for long sentences
                                                                                                                                                                                                                                                                                                                                                                                                    "spelling_error_ratio": spelling_error_ratio,
                                                                                                                                                                                                                                                                                                                                                                                                            "ttr": ttr,
                                                                                                                                                                                                                                                                                                                                                                                                                    "function_word_ratio": function_word_ratio,
                                                                                                                                                                                                                                                                                                                                                                                                                            "discourse_ratio": discourse_ratio,
                                                                                                                                                                                                                                                                                                                                                                                                                                    # Meta
                                                                                                                                                                                                                                                                                                                                                                                                                                            "sentence_length": sentence_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                    "avg_word_length": avg_word_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                            # Interaction terms — let the model learn which to trust
                                                                                                                                                                                                                                                                                                                                                                                                                                                                    "function_count_x_length": function_count_x_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                            "discourse_count_x_length": discourse_count_x_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    "function_ratio_x_length": function_ratio_x_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            "discourse_ratio_x_length": discourse_ratio_x_length,
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                }

In [33]:
df_feat['features'] = df_feat['answers'].apply(extract_features)

In [38]:
df_feat['features'].head()

0    {'chat_word_count': 2, 'punct_count': 48, 'fun...
1    {'chat_word_count': 1, 'punct_count': 46, 'fun...
2    {'chat_word_count': 4, 'punct_count': 55, 'fun...
3    {'chat_word_count': 1, 'punct_count': 20, 'fun...
4    {'chat_word_count': 1, 'punct_count': 29, 'fun...
Name: features, dtype: object

In [39]:
features_df = pd.DataFrame(df_feat['features'].tolist())

In [ ]:
features_df.head()

,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length,avg_word_length,function_count_x_length,discourse_count_x_length,function_ratio_x_length,discourse_ratio_x_length
0,2,48,118,15,1.0,0.524038,0.441948,0.056180,267,4.230769,31506,4005,118.0,15.0
1,1,46,192,24,1.0,0.527933,0.462651,0.057831,415,4.435754,79680,9960,192.0,24.0
2,4,55,196,24,1.0,0.483791,0.408333,0.050000,480,4.655860,94080,11520,196.0,24.0
3,1,20,92,12,1.0,0.670659,0.462312,0.060302,199,4.616766,18308,2388,92.0,12.0
4,1,29,147,15,1.0,0.609562,0.515789,0.052632,285,4.669323,41895,4275,147.0,15.0


In [41]:
features_df.to_pickle('features.pkl')

In [50]:
df_feat = pd.concat([df_feat, features_df], axis=1)
df_feat.head()


,answers,label,chat_word_count,punct_count,function_count,discourse_count,spelling_error_ratio,ttr,function_word_ratio,discourse_ratio,sentence_length,avg_word_length,function_count_x_length,discourse_count_x_length,function_ratio_x_length,discourse_ratio_x_length
0,"basically there are many categories of "" best ...",0,2,48,118,15,1.0,0.524038,0.441948,0.056180,267,4.230769,31506,4005,118.0,15.0
1,salt is good for not dying in car crashes and ...,0,1,46,192,24,1.0,0.527933,0.462651,0.057831,415,4.435754,79680,9960,192.0,24.0
2,the way it works is that old tv stations got a...,0,4,55,196,24,1.0,0.483791,0.408333,0.050000,480,4.655860,94080,11520,196.0,24.0
3,you ca n't just go around assassinating the le...,0,1,20,92,12,1.0,0.670659,0.462312,0.060302,199,4.616766,18308,2388,92.0,12.0
4,wanting to kill the shit out of germans drives...,0,1,29,147,15,1.0,0.609562,0.515789,0.052632,285,4.669323,41895,4275,147.0,15.0


Index(['answers', 'label'], dtype='str')